In [1]:
import os
import glob
import json
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
from scipy.stats import spearmanr

file_path = r"D:\OneDrive\Trading\Prediction Markets\data\20260731_180238\yes_markets_000000.parquet"

market = pd.read_parquet(file_path)

market

,condition_id,question,description,volume,volumeNum,liquidityNum,orderPriceMinTickSize,orderMinSize,best_bid,best_ask,spread,orderbook,events,taker_fee_rate,yes_price,timestamp,token_id
0,0xb122b2a17c8dea01d5e8ffe04316bbe011f9bf80c989...,Will Edmundo González be the leader of Venezue...,This market will resolve to the individual who...,1196813.9202239998,1.196814e+06,99790.83912,0.001,5,0.001,0.999,0.998,"{""market"": ""0xb122b2a17c8dea01d5e8ffe04316bbe0...","[{""id"": ""143443"", ""ticker"": ""venezuela-leader-...",0.04,0.0045,1.785492e+09,8548278993674426031199937135069998704424764430...
1,0xe70c3428d7d39579c7357aacc1fa31687e0323f782e1...,Israel x Iran ceasefire continues through July...,"This market will resolve to ""Yes"" if a state o...",549196.3691769999,5.491964e+05,99789.66492,0.001,5,0.001,0.999,0.998,"{""market"": ""0xe70c3428d7d39579c7357aacc1fa3168...","[{""id"": ""711714"", ""ticker"": ""israel-x-iran-cea...",0.00,0.9985,1.785492e+09,4378372829775100304973119778449304771221388117...
2,0x77fd04193f9072f3d4d518d6d10fb8cced5216bb8f6f...,Dota 2: Vici Gaming vs Amaru Gaming (BO3) - Ga...,This market refers to the Dota 2 match between...,8162.534159000001,8.162534e+03,99747.35430,0.010,5,0.010,0.990,0.980,"{""market"": ""0x77fd04193f9072f3d4d518d6d10fb8cc...","[{""id"": ""762332"", ""ticker"": ""dota2-vg-amaru-20...",0.05,0.7200,1.785492e+09,8743312876371548299993803411208410072759702994...
3,0x0d6642ddd35287eb369945b9c2b00000b7526d341e3d...,Will Tampa Bay Buccaneers win the 2027 NFL NFC...,This market will resolve according to the team...,688141.441665,6.881414e+05,99669.67146,0.001,5,0.001,0.999,0.998,"{""market"": ""0x0d6642ddd35287eb369945b9c2b00000...","[{""id"": ""203716"", ""ticker"": ""pro-football-2027...",0.05,0.0330,1.785492e+09,1281780735627751541012979933001907588268521803...
4,0x0cfec4bdb5b2060bba705259a76c489a9b0cc36da5ed...,Will Julian Assange win the Nobel Peace Prize ...,This market will resolve according to the winn...,848617.693189,8.486177e+05,99594.98961,0.001,5,0.001,0.999,0.998,"{""market"": ""0x0cfec4bdb5b2060bba705259a76c489a...","[{""id"": ""60182"", ""ticker"": ""nobel-peace-prize-...",0.05,0.0160,1.785492e+09,2514203837854735205913781359327052377275050226...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,0xcf4cf0cf9882b50018054826d561f688e319e6e51de9...,Will Carlos Alcaraz win the 2026 Men's US Open?,The 2026 U.S. Open tennis tournament is schedu...,152942.603409,1.529426e+05,87301.50150,0.010,5,0.010,0.990,0.980,"{""market"": ""0xcf4cf0cf9882b50018054826d561f688...","[{""id"": ""139236"", ""ticker"": ""2026-mens-us-open...",0.05,0.2200,1.785492e+09,7140190401024742775432978427792420584182046086...
76,0xc56ca6f0c6d37a467d3e0378ab2401763d9b3ed52169...,UFC Fight Night: Jan Blachowicz vs. Navajo Sti...,"This market will resolve to ""Jan Blachowicz"" i...",19446.233325,1.944623e+04,87123.20500,0.010,5,0.010,0.990,0.980,"{""market"": ""0xc56ca6f0c6d37a467d3e0378ab240176...","[{""id"": ""723198"", ""ticker"": ""ufc-jan-nav-2026-...",0.05,0.2650,1.785492e+09,8268803556501148671099833031721938221722441351...
77,0xa5d79e71e66c9fe122f8b8b3ca6ab7a0e3048bd8508f...,Will Xavier Becerra win the California Governo...,This market will resolve to according to the c...,1709122.5987630007,1.709123e+06,86497.69746,0.001,5,0.001,0.999,0.998,"{""market"": ""0xa5d79e71e66c9fe122f8b8b3ca6ab7a0...","[{""id"": ""57096"", ""ticker"": ""california-governo...",0.04,0.9240,1.785492e+09,6097712929239688184883391036111210717441648901...
78,0x8e6cfddfadbe05667e986a40ce40b9e853dcb9507804...,Mubadala Citi DC Open: Alex de Minaur vs Brand...,This market refers to the tennis match between...,10214.223566,1.021422e+04,86442.24320,0.010,5,0.010,0.990,0.980,"{""market"": ""0x8e6cfddfadbe05667e986a40ce40b9e8...","[{""id"": ""774378"", ""ticker"": ""atp-minaur-nakash...",0.05,0.6150,1.785492e+09,6102557019147312763020361819815387266535248059...


In [3]:
market.iloc[0]["orderbook"]

'{"market": "0xb122b2a17c8dea01d5e8ffe04316bbe011f9bf80c989bb82afde2f39a3fd4441", "asset_id": "85482789936744260311999371350699987044247644306653001528276695717482676507600", "timestamp": "1785492061590", "hash": "79b0e7efdd65172bd6d5608150d72ec27f84a7c4", "bids": [{"price": "0.001", "size": "708791"}, {"price": "0.002", "size": "52600"}, {"price": "0.003", "size": "7524.46"}, {"price": "0.004", "size": "14736.95"}], "asks": [{"price": "0.999", "size": "294604.17"}, {"price": "0.998", "size": "1138750.02"}, {"price": "0.997", "size": "97812.67"}, {"price": "0.996", "size": "61562.5"}, {"price": "0.995", "size": "34375"}, {"price": "0.994", "size": "26197.93"}, {"price": "0.993", "size": "18883.95"}, {"price": "0.992", "size": "14765.62"}, {"price": "0.991", "size": "11736.11"}, {"price": "0.99", "size": "28944.48"}, {"price": "0.989", "size": "50000"}, {"price": "0.988", "size": "10000"}, {"price": "0.98", "size": "12609.26"}, {"price": "0.979", "size": "5000"}, {"price": "0.97", "size

In [ ]:
folder_path = r"D:\OneDrive\Trading\Prediction Markets\data\20260731_180238\yes_markets_000000.parquet"

def generate_events():
    
    def chunk_id(path):
        return int(os.path.basename(path).split("_")[1].split(".")[0])

    events_files = sorted(glob.glob(os.path.join(folder_path, "events_*.parquet")), key = chunk_id)
    events = pd.concat((pd.read_parquet(f) for f in events_files), ignore_index=True)

    events = events.sort_values("local_ts").copy()
    events.to_parquet(os.path.join(folder_path, "events.parquet"), index=False)
    return events

events = generate_events()
events